<a href="https://colab.research.google.com/github/MMarcimiano/Big_Data_PJBL2-Spark/blob/main/PJBL2_Big_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

tar: spark-3.5.1-bin-hadoop3.tgz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PJBL2") \
    .getOrCreate()

sc = spark.sparkContext

In [7]:
arquivo = "/content/drive/MyDrive/BigData/operacoes_comerciais_inteira.csv"

rdd = sc.textFile(arquivo)

# Cabeçalho
cabecalho = rdd.first()

# Remove cabeçalho
rdd_sem_cabecalho = rdd.filter(
    lambda linha: linha != cabecalho
)

# Divide colunas
rdd_colunas = rdd_sem_cabecalho.map(
    lambda linha: linha.split(";")
)

# Mantém apenas registros válidos
rdd_limpo = rdd_colunas.filter(
    lambda campos:
        len(campos) == 10 and
        campos[0].strip() != "" and
        campos[1].strip() != ""
)

# Exercício 1
# Número de transações envolvendo o Brasil

In [8]:
resultado = (
    rdd_limpo
    .filter(lambda campos: campos[0].strip() == "Brazil")
    .count()
)

print("Número de transações envolvendo o Brasil:")
print(resultado)

Número de transações envolvendo o Brasil:
184748


In [9]:
import os

os.makedirs("resultados", exist_ok=True)

In [10]:
with open("resultados/exercicio1.txt", "w", encoding="utf-8") as arquivo_saida:
    arquivo_saida.write(str(resultado))

print("Resultado salvo em resultados/exercicio1.txt")

Resultado salvo em resultados/exercicio1.txt


# Exercício 2
# Número de transações envolvendo o Brasil

In [11]:
resultado = (
    rdd_limpo
    .filter(lambda campos: campos[0].strip() == "Brazil")
    .map(lambda campos: ("Brazil", 1))
    .reduceByKey(lambda a, b: a + b)
)

resultado_final = resultado.collect()

print("Resultado:")
for pais, quantidade in resultado_final:
    print(f"{pais}: {quantidade}")

Resultado:
Brazil: 184748


In [12]:
with open("resultados/exercicio2.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Número de transações envolvendo o Brasil (PairRDD)\n\n"
    )

    for pais, quantidade in resultado_final:
        arquivo_saida.write(
            f"{pais}\t{quantidade}\n"
        )

print("Resultado salvo em resultados/exercicio2.txt")

Resultado salvo em resultados/exercicio2.txt


# Exercício 3
# Número de transações envolvendo o Brasil durante 2016

In [13]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[0].strip() == "Brazil" and
            campos[1].strip() == "2016"
    )
    .count()
)

print("Número de transações do Brasil em 2016:")
print(resultado)

Número de transações do Brasil em 2016:
6210


In [14]:
with open("resultados/exercicio3.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Número de transações envolvendo o Brasil em 2016\n\n"
    )

    arquivo_saida.write(str(resultado))

print("Resultado salvo em resultados/exercicio3.txt")

Resultado salvo em resultados/exercicio3.txt


# Exercício 4
# Número de transações envolvendo o Brasil durante 2016 (PairRDD)

In [15]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[0].strip() == "Brazil" and
            campos[1].strip() == "2016"
    )
    .map(
        lambda campos:
            (("Brazil", "2016"), 1)
    )
    .reduceByKey(lambda a, b: a + b)
)

resultado_final = resultado.collect()

print("Resultado:")
for chave, quantidade in resultado_final:
    print(f"{chave}: {quantidade}")

Resultado:
('Brazil', '2016'): 6210


In [16]:
with open("resultados/exercicio4.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Número de transações envolvendo o Brasil em 2016 (PairRDD)\n\n"
    )

    for chave, quantidade in resultado_final:

        pais, ano = chave

        arquivo_saida.write(
            f"{pais}\t{ano}\t{quantidade}\n"
        )

print("Resultado salvo em resultados/exercicio4.txt")

Resultado salvo em resultados/exercicio4.txt


# Exercício 5
# Número de transações por flow e por ano ordenados por ano a partir de 2010

In [17]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[1].isdigit() and
            int(campos[1]) >= 2010 and
            campos[4].strip() != ""
    )
    .map(
        lambda campos:
            (
                (
                    int(campos[1]),
                    campos[4].strip().upper()
                ),
                1
            )
    )
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: (x[0][0], x[0][1]))
)

resultado_final = resultado.collect()

print("Primeiros resultados:")
for chave, quantidade in resultado_final[:20]:
    print(chave, quantidade)

Primeiros resultados:
(2010, 'EXPORT') 125013
(2010, 'IMPORT') 222447
(2010, 'RE-EXPORT') 17058
(2010, 'RE-IMPORT') 9273
(2011, 'EXPORT') 125909
(2011, 'IMPORT') 220850
(2011, 'RE-EXPORT') 17564
(2011, 'RE-IMPORT') 10273
(2012, 'EXPORT') 128072
(2012, 'IMPORT') 222034
(2012, 'RE-EXPORT') 16900
(2012, 'RE-IMPORT') 10337
(2013, 'EXPORT') 127904
(2013, 'IMPORT') 216282
(2013, 'RE-EXPORT') 16758
(2013, 'RE-IMPORT') 9992
(2014, 'EXPORT') 125142
(2014, 'IMPORT') 208749
(2014, 'RE-EXPORT') 20139
(2014, 'RE-IMPORT') 10424


In [18]:
with open("resultados/exercicio5.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Ano\tFlow\tQuantidade_Transacoes\n"
    )

    for chave, quantidade in resultado_final:

        ano, flow = chave

        arquivo_saida.write(
            f"{ano}\t{flow}\t{quantidade}\n"
        )

print("Resultado salvo em resultados/exercicio5.txt")

Resultado salvo em resultados/exercicio5.txt


# Exercício 6
# Média da coluna Price (trade_usd) para o ano de 2016

In [19]:
precos_2016 = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[1].strip() == "2016" and
            campos[5].strip() != ""
    )
    .map(
        lambda campos:
            float(campos[5])
    )
)

soma = precos_2016.sum()

quantidade = precos_2016.count()

media = soma / quantidade

print("Média da coluna Price (trade_usd) em 2016:")
print(f"{media:.2f}")

Média da coluna Price (trade_usd) em 2016:
137984606.20


In [20]:
with open("resultados/exercicio6.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        f"{media:.2f}"
    )

print("Resultado salvo em resultados/exercicio6.txt")

Resultado salvo em resultados/exercicio6.txt


# Exercício 7
# Média da coluna Price (trade_usd) para o ano de 2016

In [21]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[1].strip() == "2016" and
            campos[5].strip() != ""
    )
    .map(
        lambda campos:
            (
                int(campos[1]),
                (float(campos[5]), 1)
            )
    )
    .reduceByKey(
        lambda a, b:
            (
                a[0] + b[0],
                a[1] + b[1]
            )
    )
    .mapValues(
        lambda x:
            x[0] / x[1]
    )
)

resultado_final = resultado.collect()

print("Resultado:")
for ano, media in resultado_final:
    print(f"{ano}: {media:.2f}")

Resultado:
2016: 137984606.20


In [22]:
with open("resultados/exercicio7.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Ano\tMedia_Price\n"
    )

    for ano, media in resultado_final:

        arquivo_saida.write(
            f"{ano}\t{media:.2f}\n"
        )

print("Resultado salvo em resultados/exercicio7.txt")

Resultado salvo em resultados/exercicio7.txt


# Exercício 8
# Preço máximo e mínimo por categoria e por ano

In [23]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[1].strip() != "" and
            campos[5].strip() != "" and
            campos[9].strip() != ""
    )
    .map(
        lambda campos:
            (
                (
                    int(campos[1]),
                    campos[9].strip().upper()
                ),
                (
                    float(campos[5]),
                    float(campos[5])
                )
            )
    )
    .reduceByKey(
        lambda a, b:
            (
                max(a[0], b[0]),
                min(a[1], b[1])
            )
    )
    .sortBy(
        lambda x:
            (x[0][0], x[0][1])
    )
)

resultado_final = resultado.collect()

print("Primeiros resultados:")

for chave, valores in resultado_final[:20]:

    ano, categoria = chave
    maximo, minimo = valores

    print(
        f"{ano} | {categoria} | "
        f"MAX={maximo:.2f} | MIN={minimo:.2f}"
    )

Primeiros resultados:
1988 | 01_LIVE_ANIMALS | MAX=180808372.00 | MIN=109.00
1988 | 02_MEAT_AND_EDIBLE_MEAT_OFFAL | MAX=1495935657.00 | MIN=23.00
1988 | 03_FISH_CRUSTACEANS_MOLLUSCS_AQUATIC_INVERTEBRATES_NE | MAX=157683193.00 | MIN=18.00
1988 | 04_DAIRY_PRODUCTS_EGGS_HONEY_EDIBLE_ANIMAL_PRODUCT_NES | MAX=1474784000.00 | MIN=12.00
1988 | 05_PRODUCTS_OF_ANIMAL_ORIGIN_NES | MAX=222282000.00 | MIN=7.00
1988 | 06_LIVE_TREES_PLANTS_BULBS_ROOTS_CUT_FLOWERS_ETC | MAX=867214976.00 | MIN=25.00
1988 | 07_EDIBLE_VEGETABLES_AND_CERTAIN_ROOTS_AND_TUBERS | MAX=443305984.00 | MIN=46.00
1988 | 08_EDIBLE_FRUIT_NUTS_PEEL_OF_CITRUS_FRUIT_MELONS | MAX=455224992.00 | MIN=2.00
1988 | 09_COFFEE_TEA_MATE_AND_SPICES | MAX=1853880064.00 | MIN=8.00
1988 | 10_CEREALS | MAX=2086840530.00 | MIN=7.00
1988 | 11_MILLING_PRODUCTS_MALT_STARCHES_INULIN_WHEAT_GLUTE | MAX=218618703.00 | MIN=26.00
1988 | 12_OIL_SEED_OLEAGIC_FRUITS_GRAIN_SEED_FRUIT_ETC_NE | MAX=1427609024.00 | MIN=25.00
1988 | 13_LAC_GUMS_RESINS_VEGETABLE_SAP

In [24]:
with open("resultados/exercicio8.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Ano\tCategoria\tPreco_Maximo\tPreco_Minimo\n"
    )

    for chave, valores in resultado_final:

        ano, categoria = chave
        maximo, minimo = valores

        arquivo_saida.write(
            f"{ano}\t{categoria}\t"
            f"{maximo:.2f}\t"
            f"{minimo:.2f}\n"
        )

print("Resultado salvo em resultados/exercicio8.txt")

Resultado salvo em resultados/exercicio8.txt


# Exercício 9
# País com a maior exportação

In [25]:
exportacoes_por_pais = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[4].strip().upper() == "EXPORT" and
            campos[5].strip() != ""
    )
    .map(
        lambda campos:
            (
                campos[0].strip(),      # país
                float(campos[5])        # trade_usd
            )
    )
    .reduceByKey(lambda a, b: a + b)
)

pais_maior_exportacao = exportacoes_por_pais.max(
    key=lambda x: x[1]
)

resultado_final = [
    (
        (
            pais_maior_exportacao[0],
            "EXPORT"
        ),
        pais_maior_exportacao[1]
    )
]

print("Resultado:")
for chave, valor in resultado_final:
    print(f"{chave} -> {valor:.2f}")

Resultado:
('EU-28', 'EXPORT') -> 37135546884367.00


In [26]:
with open("resultados/exercicio9.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Pais\tFlow\tValor_Exportacao\n"
    )

    for chave, valor in resultado_final:

        pais, flow = chave

        arquivo_saida.write(
            f"{pais}\t{flow}\t{valor:.2f}\n"
        )

print("Resultado salvo em resultados/exercicio9.txt")

Resultado salvo em resultados/exercicio9.txt


# Exercício 10
# Preço mínimo por país e por ano

In [27]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[0].strip() != "" and
            campos[1].strip() != "" and
            campos[5].strip() != ""
    )
    .map(
        lambda campos:
            (
                (
                    campos[0].strip(),      # país
                    int(campos[1])          # ano
                ),
                float(campos[5])            # preço
            )
    )
    .reduceByKey(min)
    .sortBy(
        lambda x:
            (
                x[0][1],    # ano
                x[0][0]     # país
            )
    )
)

resultado_final = resultado.collect()

print("Primeiros resultados:")

for chave, preco_minimo in resultado_final[:20]:

    pais, ano = chave

    print(
        f"{ano} | {pais} | {preco_minimo:.2f}"
    )

Primeiros resultados:
1988 | Australia | 7.00
1988 | Finland | 2.00
1988 | Fmr Fed. Rep. of Germany | 1000.00
1988 | Greece | 2.00
1988 | Haiti | 101.00
1988 | Iceland | 8.00
1988 | India | 6.00
1988 | Japan | 1607.00
1988 | Portugal | 5.00
1988 | Rep. of Korea | 1.00
1988 | Switzerland | 7.00
1988 | Thailand | 4.00
1989 | Australia | 1.00
1989 | Bangladesh | 1.00
1989 | Brazil | 1.00
1989 | Canada | 4.00
1989 | Cyprus | 2.00
1989 | Denmark | 95.00
1989 | Finland | 7.00
1989 | Fmr Fed. Rep. of Germany | 1000.00


In [28]:
with open("resultados/exercicio10.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Ano\tPais\tPreco_Minimo\n"
    )

    for chave, preco_minimo in resultado_final:

        pais, ano = chave

        arquivo_saida.write(
            f"{ano}\t{pais}\t{preco_minimo:.2f}\n"
        )

print("Resultado salvo em resultados/exercicio10.txt")

Resultado salvo em resultados/exercicio10.txt


# Exercício 11
# Transação com maior preço por kg do tipo Export

In [29]:
resultado = (
    rdd_limpo
    .filter(
        lambda campos:
            campos[4].strip().upper() == "EXPORT" and
            campos[5].strip() != "" and
            campos[6].strip() != "" and
            float(campos[6]) > 0
    )
    .map(
        lambda campos:
            (
                float(campos[5]) / float(campos[6]),  # preço por kg

                (
                    int(campos[1]),      # ano
                    campos[0].strip(),   # país
                    campos[9].strip()    # categoria
                )
            )
    )
)

maior_transacao = resultado.max(
    key=lambda x: x[0]
)

preco_por_kg = maior_transacao[0]
ano, pais, categoria = maior_transacao[1]

print("Maior preço por kg encontrado:")
print(f"Ano: {ano}")
print(f"País: {pais}")
print(f"Categoria: {categoria}")
print(f"Preço por kg: {preco_por_kg:.2f}")

Maior preço por kg encontrado:
Ano: 1999
País: Norway
Categoria: 89_ships_boats_and_other_floating_structures
Preço por kg: 292668800.00


In [30]:
with open("resultados/exercicio11.txt", "w", encoding="utf-8") as arquivo_saida:

    arquivo_saida.write(
        "Ano\tPais\tCategoria\tPreco_por_Kg\n"
    )

    arquivo_saida.write(
        f"{ano}\t{pais}\t{categoria}\t{preco_por_kg:.2f}\n"
    )

print("Resultado salvo em resultados/exercicio11.txt")

Resultado salvo em resultados/exercicio11.txt


In [31]:
import shutil

shutil.make_archive(
    "resultados",
    "zip",
    "resultados"
)

'/content/resultados.zip'